# OCR Validation Notebook

Run this **before** submitting the full Spark job to:
1. Confirm EasyOCR works on the server GPU
2. Spot-check image quality and OCR output
3. Validate the path regex used by the pipeline

## 1 — Validate GPU availability

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}:', torch.cuda.get_device_name(i))

## 2 — Sample image paths

Edit the list below to point at real files from your dataset.

In [ ]:
import os
import glob

IMAGES_DIR = '/workspace/data/images'

# Auto-pick 5 images spread across subreddits
all_images = glob.glob(f'{IMAGES_DIR}/**/*.jpg', recursive=True) + \
             glob.glob(f'{IMAGES_DIR}/**/*.jpeg', recursive=True) + \
             glob.glob(f'{IMAGES_DIR}/**/*.png', recursive=True)

print(f'Total images found: {len(all_images)}')

# Sample up to 5
import random
random.seed(42)
test_paths = random.sample(all_images, min(5, len(all_images)))
print('Sampled paths:')
for p in test_paths:
    print(' ', p)

## 3 — Run EasyOCR on sample images

In [ ]:
import easyocr
import matplotlib.pyplot as plt
from PIL import Image

reader = easyocr.Reader(['en'], gpu=True)

for path in test_paths:
    if not os.path.exists(path):
        print(f'File not found: {path}')
        continue

    results = reader.readtext(path, detail=1)
    texts = [r[1] for r in results]
    confidences = [round(r[2], 2) for r in results]

    img = Image.open(path)
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.title(f"{os.path.basename(path)}\n{texts}", fontsize=9)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    print(f'Extracted : {texts}')
    print(f'Confidence: {confidences}')
    print('---')

## 4 — Validate path regex

Confirm the regex used in the Spark pipeline parses subreddit and filename correctly.

In [ ]:
import re

for path in test_paths:
    # Spark uses file:// URIs internally — strip that prefix if present for testing
    test_path = path.replace('file://', '')

    subreddit_match = re.search(r'/images/([^/]+)/', test_path)
    filename_match  = re.search(r'/([^/]+\.(?:jpe?g|png))$', test_path)

    subreddit = subreddit_match.group(1) if subreddit_match else 'NO MATCH'
    filename  = filename_match.group(1)  if filename_match  else 'NO MATCH'

    print(f'Path      : {test_path}')
    print(f'Subreddit : {subreddit}')
    print(f'Filename  : {filename}')
    print('---')

## 5 — Inspect consolidated metadata

In [ ]:
import pandas as pd

META_PATH = '/workspace/data/metadata_consolidated.csv'

if not os.path.exists(META_PATH):
    print(f'Not found: {META_PATH}')
    print('Run consolidate_metadata.py first.')
else:
    df_meta = pd.read_csv(META_PATH, nrows=5)
    print('Columns:', df_meta.columns.tolist())
    print()
    display(df_meta)